# Volume 10, Chapter 6: Stein's method (Python companion)

Exercises first, solutions at the end. We solve the Stein equation numerically and look at its derivatives, measure Wasserstein distances exactly, test the bound for sampling without replacement, and put the Chen–Stein method to work. The computer science application is **sizing random identifiers and hash values** with a certified collision probability. Requires `numpy` and `scipy`.

In [ ]:
import math
import numpy as np
from scipy import stats, integrate
rng = np.random.default_rng(60)
phi, Phi = stats.norm.pdf, stats.norm.cdf

## Exercises

**P6.1 (★) The solution of the Stein equation.** For $h(x)=|x|$, $h(x)=\sin x$ and $h(x)=\min(1,\max(0,x))$ (all with $\|h'\|=1$), compute $f(w)=\frac1{\varphi(w)}\int_{-\infty}^w(h-\mathbb Eh(Z))\varphi$ on a grid, together with $f'=wf+\tilde h$, and check the bounds of Lemma 6.2. Then check numerically that $2\psi(w)\psi(-w)/\varphi(w)$, which bounds $|f'|$, has its maximum $\sqrt{2/\pi}$ at $w=0$.

**P6.2 (★) Wasserstein distances, exactly.** For laws on $\mathbb R$, $d_W(X,Y)=\int_{\mathbb R}|F_X(x)-F_Y(x)|\,\mathrm dx$ (a classical formula; it can be checked here against the definition). Compute $d_W$ between the normalized binomial$(n,p)$ and $N(0,1)$ for $p=\frac12$ and $p=\frac1{10}$, $n=10,100,1000,10^4$. Compare with the upper bound $3\rho/\sqrt n$ of Theorem 6.4 and, for $p=\frac12$, with the lower bound of Exercise 6.6.

**P6.3 (★★) Polls without replacement.** For the hypergeometric count of Example 6.6, compute the exact $d_W$ to the normal law and the bound $2/\sigma+4/N$ for $N=10^4$, $p=\frac12$ and $m=100,1000,5000,9000$, and for $N=10^3$, $p=0.3$, $m=100$. In which regime is the binomial approximation bad while the normal one is still good?

**P6.4 (★★) CS application: how many bits for a random identifier?** A system tags each of $n$ objects with a random $b$-bit identifier (or a $b$-bit hash of its content). (a) For $b=32$, compare the exact probability of no collision with $e^{-\lambda}$ and the Chen–Stein error bound for $n=10^4,10^5,77163,10^6$. (b) Find the smallest $b$ such that the probability of at least one collision among $n=10^9$ objects is *certified* to be below $10^{-6}$, using only Example 6.9. (c) Simulate the number of colliding pairs for $n=2000$ and $d=2^{20}$ and compare its law with Poisson$(\lambda)$.

**P6.5 (★★) Triangles.** Simulate $G(n,c/n)$ for $n=300$ and $c=1.5$, $2$, $3$, count triangles, and compare the empirical law with Poisson$(\binom n3p^3)$: the estimated total variation distance against the bound of Example 6.10. (Count triangles as $\frac16\operatorname{tr}A^3$, computed as the sum of the entries of $A^2\circ A$.)

**P6.6 (★★) Long runs of heads.** For $n=10^5$ flips and $r=15,16,17$, estimate by simulation the probability that there is no run of $r$ heads, and compare with $e^{-n2^{-r-1}}$ and the bound $(2r+1)2^{-r-1}$ of Exercise 6.5. Also check the Diaconis–Mosteller near-birthday figure of Exercise 6.4 by simulation.

## Solutions

### P6.1
**How to find it.** Integrate $\tilde h\varphi$ with `quad`, divide by $\varphi(w)$, and get $f'$ from the equation itself rather than by differentiating numerically; $f''=f+wf'+h'$ likewise. Far in the tails, $\int_{-\infty}^w$ is a tiny number divided by a tiny number, so for $w>0$ use the equivalent form $-\frac1{\varphi(w)}\int_w^\infty\tilde h\varphi$.

In [ ]:
tests = {
    "|x|":    (np.abs,                          lambda x: np.sign(x)),
    "sin x":  (np.sin,                          np.cos),
    "ramp":   (lambda x: np.clip(x, 0, 1),      lambda x: ((x > 0) & (x < 1)).astype(float)),
}
ws = np.linspace(-6, 6, 1201)
for name, (h, dh) in tests.items():
    def I(fun, a, b):
        # integral over [a, b], split at the corners 0 and 1 of the test functions
        cuts = [a] + [c for c in (0.0, 1.0) if a < c < b] + [b]
        return sum(integrate.quad(fun, u, v, limit=200)[0] for u, v in zip(cuts, cuts[1:]))
    Eh = I(lambda x: h(x) * phi(x), -40, 40)
    def f(w):
        if w <= 0:
            return I(lambda x: (h(x) - Eh) * phi(x), -40, w) / phi(w)
        return -I(lambda x: (h(x) - Eh) * phi(x), w, 40) / phi(w)
    F = np.array([f(w) for w in ws])
    F1 = ws * F + h(ws) - Eh
    F2 = F + ws * F1 + dh(ws)
    print(f"{name:6s}  sup|f| = {abs(F).max():.4f}   sup|f'| = {abs(F1).max():.4f}   sup|f''| = {abs(F2).max():.4f}")
psi = lambda w: phi(w) - w * (1 - Phi(w))
g = 2 * psi(ws) * psi(-ws) / phi(ws)
print("max of 2 psi(w) psi(-w)/phi(w):", g.max(), "at w =", ws[g.argmax()], "  sqrt(2/pi) =", math.sqrt(2 / math.pi))

All three stay inside $\|f\|\le1$, $\|f'\|\le2$ (in fact $\le\sqrt{2/\pi}\approx0.798$) and $\|f''\|\le2$. The ramp and $|x|$ are only Lipschitz, so $f''$ jumps at their corners, but the bound still holds pointwise away from them. For $h(x)=|x|$ the bound $\sqrt{2/\pi}$ on $f'$ is attained at $0$ (Figure 6.1).

### P6.2
**How to find it.** The formula $d_W=\int|F_X-F_Y|$ turns the supremum over all $1$-Lipschitz functions into one integral. For a lattice law the distribution function is a step function, so integrate on a fine grid.

In [ ]:
def dW_normal(vals, probs, mu, sig, lo=-12, hi=12, M=600001):
    x = (np.asarray(vals) - mu) / sig
    grid = np.linspace(lo, hi, M)
    cdf = np.concatenate([[0], np.cumsum(probs)])[np.searchsorted(x, grid, side='right')]
    return np.trapezoid(np.abs(cdf - Phi(grid)), grid)

print("   n      p     d_W       3 rho/sqrt(n)   lattice lower bound")
for p in (0.5, 0.1):
    rho = (p * (1 - p)**3 + (1 - p) * p**3) / (p * (1 - p))**1.5
    for n in (10, 100, 1000, 10**4):
        k = np.arange(n + 1)
        d = dW_normal(k, stats.binom.pmf(k, n, p), n * p, math.sqrt(n * p * (1 - p)))
        low = 1 / (2 * math.sqrt(n)) - 4 * phi(0) / n if p == 0.5 else float('nan')
        print(f"{n:6d}  {p:4.1f}  {d:.5f}     {3 * rho / math.sqrt(n):.5f}       {low:.5f}")

For $p=\frac12$, $d_W\cdot\sqrt n$ settles near $0.5$: the lattice lower bound $\frac1{2\sqrt n}$ is essentially the truth, and the Stein bound $\frac3{\sqrt n}$ is off by a factor of six. For $p=\frac1{10}$ the skewness adds to the distance, and $\rho$ is larger, so the bound grows too. Both scale like $n^{-1/2}$, as Exercise 6.6 says they must.

### P6.3
**How to find it.** Same exact integral, with `stats.hypergeom`. The binomial approximation (sampling *with* replacement) has the wrong variance when $m/N$ is not small: it misses the factor $\frac{N-m}{N-1}$.

In [ ]:
print("    N     p     m     sigma     exact d_W   bound 2/sigma+4/N")
for N, p, m in [(10**4, .5, 100), (10**4, .5, 1000), (10**4, .5, 5000), (10**4, .5, 9000), (1000, .3, 100)]:
    K = int(N * p); k = np.arange(m + 1)
    sig = math.sqrt(m * p * (1 - p) * (N - m) / (N - 1))
    d = dW_normal(k, stats.hypergeom.pmf(k, N, K, m), m * p, sig)
    print(f"{N:6d}  {p:.1f}  {m:5d}  {sig:8.3f}    {d:.5f}     {2 / sig + 4 / N:.4f}")
N, m = 10**4, 9000
print("variance ratio hypergeometric / binomial at m/N = 0.9:", (N - m) / (N - 1))

When $9000$ of $10\,000$ ballots are counted, the binomial model overstates the variance tenfold, yet the normal approximation with the right $\sigma$ is excellent, and Example 6.6 certifies it with no condition on $m/N$. The bound is about eight times the truth: Stein bounds are honest, not tight.

### P6.4
**How to find it.** (a) The exact probability is $\prod_{k<n}(1-k/d)$; sum logarithms with `log1p`. (b) Example 6.9 gives $P(\text{collision})\le1-e^{-\lambda}+(4n-7)(1-e^{-\lambda})/d$, and $1-e^{-\lambda}\le\lambda$; increase $b$ until this is below $10^{-6}$. (Here the trivial union bound $P(W\ge1)\le\mathbb EW=\lambda$ already does the job: Chen–Stein is needed for the *other* direction, to certify that collisions are *likely*.) (c) Draw, count pairs per bucket with `np.unique`, and compare frequencies.

In [ ]:
def no_collision_exact(n, d):
    return math.exp(np.sum(np.log1p(-np.arange(n, dtype=float) / d)))

d = 2.0**32
print("      n      lambda     exact        e^-lambda     |diff|      Chen-Stein bound")
for n in (10**4, 10**5, 77163, 10**6):
    lam = n * (n - 1) / 2 / d
    ex, po = no_collision_exact(n, d), math.exp(-lam)
    print(f"{n:8d}  {lam:9.4f}  {ex:.8f}  {po:.8f}  {abs(ex - po):.2e}   {(4 * n - 7) * (1 - po) / d:.2e}")

n = 10**9
for b in range(60, 128):
    d = 2.0**b; lam = n * (n - 1) / 2 / d
    upper = -math.expm1(-lam) * (1 + (4 * n - 7) / d)
    if upper < 1e-6:
        print(f"\n(b) n = 1e9: b = {b} bits gives P(collision) <= {upper:.2e}")
        break
print("    with 64 bits, n = 1e9: lambda =", n * (n - 1) / 2 / 2.0**64,
      " P(no collision) =", math.exp(-n * (n - 1) / 2 / 2.0**64), "+-", (4 * n - 7) * -math.expm1(-n * (n - 1) / 2 / 2.0**64) / 2.0**64)

n, d, T = 2000, 2**20, 20000
lam = n * (n - 1) / 2 / d
counts = np.empty(T, dtype=int)
for t in range(T):
    _, c = np.unique(rng.integers(0, d, n), return_counts=True)
    counts[t] = (c * (c - 1) // 2).sum()
emp = np.bincount(counts, minlength=12)[:12] / T
pois = stats.poisson.pmf(np.arange(12), lam)
print(f"\n(c) lambda = {lam:.4f}")
for k in range(8):
    print(f"   P(W={k}): simulated {emp[k]:.4f}   Poisson {pois[k]:.4f}")
print("   estimated TV:", 0.5 * np.abs(np.bincount(counts) / T - stats.poisson.pmf(np.arange(counts.max() + 1), lam)).sum(),
      "   Chen-Stein bound:", (4 * n - 7) * (1 - math.exp(-lam)) / d)

With $32$ bits, a hundred thousand objects collide with probability $0.69$, certified to within $10^{-4}$: random $32$-bit identifiers are not identifiers. With $64$ bits and a billion objects, the probability of a collision is about $2.7\%$, still not negligible; the certified threshold for $10^{-6}$ is $b=79$, which is why content hashes and random UUIDs use $122$ to $256$ bits. In (c) the estimated TV distance, about $0.0055$, is below the bound $0.0065$ of Example 6.9, and most of it is Monte Carlo noise from $20\,000$ samples.

### P6.5
**How to find it.** Build a symmetric $0/1$ matrix from the upper triangle, and use $\operatorname{tr}A^3=\sum_{ij}(A^2)_{ij}A_{ij}=6T$. The bound of Example 6.10 is $(b_1+b_2)(1-e^{-\lambda})/\lambda$ with the exact $b_1,b_2$.

In [ ]:
n, T = 300, 3000
iu = np.triu_indices(n, 1)
for c in (1.5, 2.0, 3.0):
    p = c / n
    tri = np.empty(T, dtype=int)
    for t in range(T):
        A = np.zeros((n, n))
        A[iu] = rng.random(len(iu[0])) < p
        A = A + A.T
        tri[t] = round(((A @ A) * A).sum() / 6)
    C3 = math.comb(n, 3); lam = C3 * p**3
    b1 = C3 * (3 * n - 8) * p**6; b2 = C3 * 3 * (n - 3) * p**5
    K = max(tri.max(), 20) + 1
    tv = 0.5 * np.abs(np.bincount(tri, minlength=K) / T - stats.poisson.pmf(np.arange(K), lam)).sum()
    print(f"c = {c}: lambda = {lam:.4f} (limit {c**3 / 6:.4f}), mean count {tri.mean():.4f}, "
          f"P(no triangle) {np.mean(tri == 0):.4f} vs e^-lambda {math.exp(-lam):.4f}; TV est {tv:.4f}, bound {(b1 + b2) * (1 - math.exp(-lam)) / lam:.4f}")

The bound is of order $c^2/n$ and indeed small; the estimated TV distance includes Monte Carlo noise of order $T^{-1/2}$ times a few, so it cannot go much below $0.01$ with $3000$ samples. The Poisson law fits well even for $c=3$, where a typical graph has $4.5$ triangles.

### P6.6
**How to find it.** A run of $r$ heads ending at position $j$ exists iff the cumulative count of heads gains $r$ over a window of length $r$: compare `cumsum` at distance $r$. For near-birthdays on a circle, sort the birthdays and look at the circular gaps.

In [ ]:
n, T = 10**5, 1000
for r in (15, 16, 17):
    hits = 0
    for t in range(T):
        x = rng.random(n) < 0.5
        cs = np.concatenate([[0], np.cumsum(x)])
        hits += np.any(cs[r:] - cs[:-r] == r)
    lam = n * 2.0**(-r - 1)
    print(f"r = {r}: P(no run) simulated {1 - hits / T:.3f}  (+-{1.96 * math.sqrt(0.25 / T):.3f}),  e^-lambda = {math.exp(-lam):.3f},  bound {(2 * r + 1) * 2.0**(-r - 1):.1e}")

# near birthdays: 14 people, 365-day circle, no two within one day
m, T = 14, 400000
b = np.sort(rng.integers(0, 365, (T, m)), axis=1)
gaps = np.diff(np.concatenate([b, b[:, :1] + 365], axis=1), axis=1)
exact = 365 * math.comb(365 - m - 1, m - 1) * math.factorial(m - 1) / 365**m
print(f"\nnear-birthdays, 14 people: simulated {np.mean(gaps.min(axis=1) >= 2):.4f}, exact {exact:.4f}, Poisson {math.exp(-3 * math.comb(m, 2) / 365):.4f}")

The simulated probabilities agree with $e^{-\lambda}$ within the Monte Carlo error: the runs counted from position $0$ (excluded in Exercise 6.5) shift them by at most $2^{-r}$. The exact count for near-birthdays places $14$ people on the circle with all gaps at least $2$: there are $\frac{365}{14}\binom{365-14-1}{13}$ such sets, and $14!$ orders. The Poisson value $0.473$ is $0.01$ too high, well within its Chen–Stein bound.